# 01. データを確認する ― 何があって、何を作るのか

Amazon Quick × Snowflake ハンズオン（Snowflake 編）の 1 本目です。
`setup.sql` で取り込んだデータを確認し、このハンズオンで作るものを整理します。**このNotebookでは何も作りません**（確認だけです）。

| Notebook | 内容 |
|---|---|
| 01 | 取り込んだデータの確認と、このハンズオンで作るもの |
| 02 | **Cortex Analyst**：Semantic View を作り、改善前後の正答率を比べる |
| 03 | **Cortex Search**：PDF レポートを検索できるようにする |
| 04 | **Cortex Agent**：Analyst と Search を組み合わせた Agent を作る |
| 05 | MCP Server と OAuth：Amazon Quick から Agent を呼べるようにする |

**前提**：`load.sql` と `setup.sql` を実行済みであること  
**所要時間**：約10分

## シナリオ

あなたは SaaS 企業の**営業企画（Sales Operations）の担当者**です。
営業部長や現場のマネージャーから、毎日のようにこんな質問が届きます。

- 「今期の地域別の売上は？ Enterprise プランはどれくらい売れている？」
- 「**停滞している商談**は何件ある？」「**勝率**は？」
- 「レポートで推奨されている**パイプラインカバレッジ**を下回っている営業担当は誰？」

数字は Snowflake のテーブルに、**社内の基準や推奨アクションは PDF のレポート**に書かれています。
このハンズオンでは、両方をまたいで自然言語で答える **Cortex Agent** を作り、最後に **Amazon Quick** から使えるようにします。

## Step 0. セッションの準備と事前チェック

このハンズオンで使う名前を変数にまとめ、AI 機能が使えるかを最初に確かめます。

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

DB, SCHEMA = "CORTEX_QUICK_HOL_DB", "CORTEX_QUICK_HOL_SCHEMA"
HOL_ROLE, HOL_WH = "CORTEX_QUICK_HOL_ROLE", "CORTEX_QUICK_HOL_WH"

session.use_role("ACCOUNTADMIN")
session.use_warehouse(HOL_WH)
session.use_schema(f"{DB}.{SCHEMA}")
print("ロール:", session.get_current_role(), "/ ウェアハウス:", session.get_current_warehouse())
info = session.sql("""SELECT CURRENT_USER() AS U, CURRENT_REGION() AS R,
  CURRENT_ORGANIZATION_NAME() || '-' || CURRENT_ACCOUNT_NAME() AS A""").collect()[0]
print("実行ユーザー:", info["U"], "/ アカウント:", info["A"], "/ リージョン:", info["R"])

### 事前チェック：AI 機能とクロスリージョン推論

後半（Agent / Quick 連携）で詰まらないよう、**最初に** AI 機能が使えるかを確かめます。

- `CORTEX_ENABLED_CROSS_REGION` は、アカウントのリージョンに必要なモデルがないときに他リージョンで推論する設定です。アカウントのリージョンによっては Agent のモデルが見つからないため、`setup.sql` で `ANY_REGION` にしています（データ自体は移動しません）。ここでは値を確認するだけです。
- `AI_COMPLETE` を1回だけ呼び、AI 機能が使えるかを確認します（数文字の応答なので消費はごくわずかです）。

In [ ]:
v = session.sql("SHOW PARAMETERS LIKE 'CORTEX_ENABLED_CROSS_REGION' IN ACCOUNT").collect()
print("CORTEX_ENABLED_CROSS_REGION:", v[0]["value"] if v else "(不明)", "（setup.sql で ANY_REGION に設定済み）")

try:
    r = session.sql("SELECT AI_COMPLETE('claude-sonnet-4-5', 'Reply with OK only.') AS R").collect()[0]["R"]
    print("OK: AI 機能が使えます（応答:", str(r).strip()[:20], "）")
except Exception as e:
    msg = str(e)
    print("⚠️ AI_COMPLETE の呼び出しに失敗しました。モデル名の問題の可能性があります（Agent の動作は 04 で確認します）:\n", msg[:300])

## Step 1. 構造化データ：6 つのテーブル

`setup.sql` で公開 S3 から読み込んだ、営業オペレーションのサンプルデータです。まず件数を確認します（各 200 件）。

| テーブル | 内容 | 主なカラム |
|---|---|---|
| `SALES_TRANSACTIONS_FACT_TABLE` | 受注した取引（ファクト） | 地域、金額、割引率、プラン SKU |
| `CUSTOMER_DIMENSION_TABLE` | 顧客 | 業種、規模、LTV、解約リスク（0〜100）、拡大余地 |
| `PRODUCT_CATALOG_TABLE` | 製品カタログ | カテゴリ、ティア、定価、更新率、アップセル率 |
| `SALES_REP_PERFORMANCE_TABLE` | 営業担当の実績 | テリトリー、目標達成率、勝率、パイプラインカバレッジ |
| `SALES_PIPELINE_OPPORTUNITIES` | 商談パイプライン | ステージ、見込金額、滞留日数、競合有無、決裁者接触 |
| `MARKETING_ATTRIBUTION_TABLE` | マーケティング施策 | キャンペーン、チャネル、リードスコア、費用 |

In [ ]:
%%sql -r row_counts
SELECT 'SALES_TRANSACTIONS_FACT_TABLE' AS TABLE_NAME, COUNT(*) AS ROW_COUNT FROM CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_TRANSACTIONS_FACT_TABLE
UNION ALL SELECT 'CUSTOMER_DIMENSION_TABLE',     COUNT(*) FROM CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.CUSTOMER_DIMENSION_TABLE
UNION ALL SELECT 'PRODUCT_CATALOG_TABLE',        COUNT(*) FROM CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.PRODUCT_CATALOG_TABLE
UNION ALL SELECT 'SALES_REP_PERFORMANCE_TABLE',  COUNT(*) FROM CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_REP_PERFORMANCE_TABLE
UNION ALL SELECT 'SALES_PIPELINE_OPPORTUNITIES', COUNT(*) FROM CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_PIPELINE_OPPORTUNITIES
UNION ALL SELECT 'MARKETING_ATTRIBUTION_TABLE',  COUNT(*) FROM CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.MARKETING_ATTRIBUTION_TABLE

### 中身を見てみる

取引と商談を数行ずつ表示します。どんなカラムにどんな値が入っているかを眺めてください。

In [ ]:
%%sql -r sample_transactions
SELECT * FROM CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_TRANSACTIONS_FACT_TABLE ORDER BY TRANSACTION_DATE LIMIT 5

In [ ]:
%%sql -r sample_pipeline
SELECT OPPORTUNITY_ID, CURRENT_STAGE, ESTIMATED_VALUE, DAYS_IN_CURRENT_STAGE,
       COMPETITOR_PRESENT, DECISION_MAKER_ENGAGED
FROM CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_PIPELINE_OPPORTUNITIES ORDER BY ESTIMATED_VALUE DESC LIMIT 5

### ざっくり集計してみる

あとで Agent に聞く質問の「正解」を、SQL で先に確かめておきます。

In [ ]:
%%sql -r revenue_by_region
SELECT REGION, COUNT(*) AS DEALS, SUM(DEAL_SIZE_USD) AS REVENUE
FROM CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_TRANSACTIONS_FACT_TABLE GROUP BY REGION ORDER BY REVENUE DESC

In [ ]:
%%sql -r pipeline_by_stage
SELECT CURRENT_STAGE, COUNT(*) AS OPPS, SUM(ESTIMATED_VALUE) AS VALUE,
       ROUND(AVG(DAYS_IN_CURRENT_STAGE), 1) AS AVG_DAYS_IN_STAGE
FROM CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_PIPELINE_OPPORTUNITIES GROUP BY CURRENT_STAGE ORDER BY VALUE DESC

> 💡 **気づいてほしいこと**
> - 解約リスク `CHURN_RISK_SCORE` は **0〜100** の値です（0〜1 ではありません）。
> - 商談名 `OPPORTUNITY_NAME` は、元データの時点で全件が空です。商談は `OPPORTUNITY_ID` で区別します。
> - 「停滞商談」「勝率」「推奨カバレッジ」といった**社内用語は、どのテーブルにもカラムとして存在しません**。
>   人間は「あの基準のことだな」と分かりますが、AI は分かりません。これを教えるのが 02 の Semantic View です。

## Step 2. データの落とし穴：取引と製品がつながらない

取引の `PRODUCT_SKU` と製品カタログの `PRODUCT_SKU` で結合してみます。

In [ ]:
%%sql -r join_check
SELECT
  (SELECT COUNT(*) FROM CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_TRANSACTIONS_FACT_TABLE t JOIN CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.CUSTOMER_DIMENSION_TABLE c ON t.CUSTOMER_ID = c.CUSTOMER_ID)     AS TXN_TO_CUSTOMER,
  (SELECT COUNT(*) FROM CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_TRANSACTIONS_FACT_TABLE t JOIN CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_REP_PERFORMANCE_TABLE r ON t.SALES_REP_ID = r.SALES_REP_ID) AS TXN_TO_REP,
  (SELECT COUNT(*) FROM CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_TRANSACTIONS_FACT_TABLE t JOIN CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.PRODUCT_CATALOG_TABLE p ON t.PRODUCT_SKU = p.PRODUCT_SKU)        AS TXN_TO_PRODUCT_CATALOG,
  (SELECT COUNT(*) FROM CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_TRANSACTIONS_FACT_TABLE t JOIN CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.PRODUCT_PLAN_VIEW p ON t.PRODUCT_SKU = p.PRODUCT_SKU)            AS TXN_TO_PRODUCT_PLAN

顧客・営業担当は 200 件すべて結合できますが、**製品カタログとは 0 件**です。
取引の SKU は `ENT-ANNUAL` のような「ティア＋請求サイクル」（6種類）、カタログは `ENT-ANN-011` のような個別製品（200種類）で、形式が違うためです（元教材のデータのままです）。

そこで `setup.sql` では、**プラン（ティア×請求サイクル）単位のビュー `PRODUCT_PLAN_VIEW`** を作っています。これなら 200 件すべて結合できます（`TXN_TO_PRODUCT_PLAN = 200`）。
このように「AI に渡す前にデータのつながりを確かめる」ことが、精度の土台になります。

In [ ]:
%%sql -r product_plan_view
SELECT * FROM CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.PRODUCT_PLAN_VIEW ORDER BY PRODUCT_SKU

## Step 3. 非構造化データ：PDF レポート 2 本

数字以外の「社内の基準」や「推奨アクション」は、PDF のレポートに書かれています。`setup.sql` で社内ステージ `DOCS_STAGE` にコピー済みです。

| PDF | 主な内容 |
|---|---|
| Sales Performance Analysis | 地域・営業担当の分析、**推奨パイプラインカバレッジ 1.6 倍**、値引きの上限の目安 |
| Customer Intelligence & Pipeline | 解約リスクの高い顧客、**停滞商談（40日超）**と推奨アクション、施策の優先順位 |

In [ ]:
%%sql -r list_docs
SELECT RELATIVE_PATH, SIZE, LAST_MODIFIED FROM DIRECTORY(@CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.DOCS_STAGE)

## このハンズオンで作るもの

| 種類 | 名前 | 役割 | 作る Notebook |
|---|---|---|---|
| Semantic View（改善前） | `SALES_OPS_SV_BASIC` | テーブルを並べただけの最小構成 | 02 |
| Semantic View（改善版） | `SALES_OPS_SEMANTIC_VIEW` | 説明・同義語・社内 KPI・検証済みクエリ入り | 02 |
| Cortex Search Service | `SALES_DOCS_SEARCH` | PDF レポートの意味検索 | 03 |
| Cortex Agent | `SALES_OPS_AGENT` | Analyst（数字）と Search（文書）を使い分けて答える | 04 |
| MCP Server / OAuth | `SALES_OPS_MCP_SERVER` / `SALES_OPS_QUICK_SUITE_OAUTH` | Amazon Quick から呼ぶ窓口 | 05 |

✅ 01 はここまでです。次は `02_cortex_analyst.ipynb` に進みます。